# 5 · Detailed Stats  `[EVAL]`

The **full statistical tables** behind the figures in `0`-`1` — kept here so the analysis notebooks stay figure-led. Everything is full-conversation eval, paired by the 96 shared personas. Thin arms (<3 scored iters) are dropped to avoid NaN rows. For a reader who wants exact numbers.

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath("."))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)
import exp3
from exp3 import stats, behavior, training, pref, figures, plots

# ── EDA config — edit these flat globals to control the whole notebook ─────────
cfg = exp3.EdaConfig(
    methods=None, ks=None, arm_labels=None,        # arm filter (None = all on disk)
    metrics=None, selection="all", warmth_only=False,  # metric subset / cross-model selection
    focus_arms=None, focus_metric="Q1Q2",          # default arms/metric for overlay & contrast figs
    panel=None, ncols=None, score_ylim=None, share_y=False,  # plot scales (None = inherit)
    export_group="stats",                         # results/<figures|tables>/stats/  ·  figs=PNG, tables=md+xlsx
)
S = exp3.notebook_setup(cfg)
FOCUS = cfg.focus_arms or sorted(S.SCORES.arm.unique())   # arms to show in overlay/contrast figures

## 1 · Main results — each arm vs its own base  `[EVAL]`
**Purpose.** Per (arm × rubric): final (and best) iteration vs base — Δ, paired Cohen's *dz* + label, Wilcoxon *p* (Holm), bootstrap 95% CI, trajectory ρ / slope.

In [ ]:
MR = stats.filter_thin_arms(stats.main_results_table(S.SCORES, target="final"), S.SCORES)
display(MR)
exp3.save_table(MR, "main_results_final", caption="Final iteration vs base, per arm x rubric, persona-paired (N=96): dz, Wilcoxon p (Holm), bootstrap CI, trajectory rho/slope. Thin arms dropped.")
MRb = stats.filter_thin_arms(stats.main_results_table(S.SCORES, target="best"), S.SCORES)
exp3.save_table(MRb, "main_results_best", caption="As main_results_final but vs each arm's BEST iteration (selection-sensitivity companion).")

## 2 · Repeated-measures omnibus (Friedman)  `[EVAL]`
**Purpose.** Is iteration a real within-persona factor? Friedman χ² + Kendall's *W* per arm × rubric.

In [ ]:
FR = pd.DataFrame([stats.friedman_trajectory(S.SCORES, a, m)
                   for a in sorted(S.SCORES.arm.unique()) for m in S.METRICS])
FR = stats.filter_thin_arms(FR, S.SCORES)
display(FR.round(4))
exp3.save_table(FR.round(4), "friedman_omnibus", caption="Friedman repeated-measures omnibus across iterations per arm x rubric (Kendall's W). Thin arms dropped.")

## 3 · Per-arm vs-base, every iteration (paired)  `[EVAL]`
**Purpose.** The full iteration-by-iteration Q1+Q2 vs-base table per arm (each arm vs its OWN base).

In [ ]:
for arm in sorted(S.SCORES.arm.unique()):
    if arm in stats.thin_arms(S.SCORES): continue
    PV = stats.paired_vs_base(S.SCORES, arm, "Q1Q2")
    if PV.empty: continue
    print(f"=== {arm}: Q1+Q2 vs base (persona-paired) ==="); display(PV[["iteration","n","mean_delta","dz","p_holm","ci_low","ci_high"]].round(4))
    exp3.save_table(PV[["iteration","n","mean_delta","dz","p","p_holm","ci_low","ci_high"]].round(4), f"{arm}_Q1Q2_vs_base_paired", caption=f"{arm} each iteration vs base on Q1+Q2; persona-paired Wilcoxon, dz, Holm p, bootstrap CI.")

## 4 · Cross-arm contrasts — PTO vs GRPO (matched K) & K0 vs K5  `[EVAL]`
**Purpose.** The method and look-ahead contrasts as paired tables at matched iterations (iteration-0 base rows dropped). Positive `mean_delta` = first term higher.

In [ ]:
for K in sorted(S.SCORES.K.unique()):
    CMP = stats.paired_method_comparison(S.SCORES, "PTO", "GRPO", K=K)
    CMP = CMP[CMP.iteration > 0] if not CMP.empty else CMP
    if CMP.empty: print(f"K={K}: no common PTO/GRPO iters"); continue
    view = CMP[["iteration","metric","n","mean_delta","dz","p_holm"]].round(4)
    print(f"=== PTO_LA{K} - GRPO_LA{K} (paired; + => PTO higher) ==="); display(view)
    exp3.save_table(view, f"PTO_vs_GRPO_LA{K}_paired", caption=f"PTO_LA{K} - GRPO_LA{K} at matched iterations; persona-paired Wilcoxon + dz + Holm.")
for method in ["PTO", "GRPO"]:
    CMP = stats.paired_k_comparison(S.SCORES, method)
    CMP = CMP[CMP.iteration > 0] if not CMP.empty else CMP
    if CMP.empty: print(f"{method}: K0-vs-K5 not comparable yet"); continue
    view = CMP[["iteration","metric","mean_delta","dz","p_holm"]].round(4)
    print(f"=== {method}: LA0 - LA5 (+ => K0 higher) ==="); display(view)
    exp3.save_table(view, f"{method}_K0_vs_K5_paired", caption=f"{method} K0 - K5 at matched iterations; persona-paired Wilcoxon + dz + Holm.")

## 5 · Climb rate, rankings, factor structure  `[EVAL]`
**Purpose.** Q1+Q2 OLS slope + Spearman ρ per arm (climb rate vs endpoint); the cross-model leaderboard; and the rubric PCA (PC1 share → are the 6 rubrics ~one factor?).

In [ ]:
SL = stats.filter_thin_arms(pd.DataFrame([stats.trajectory_test(S.SCORES, a, "Q1Q2") for a in sorted(S.SCORES.arm.unique())]), S.SCORES)
display(SL[["arm","metric","spearman_rho","ols_slope","peak_iter","final_iter"]].round(4))
exp3.save_table(SL[["arm","metric","spearman_rho","ols_slope","peak_iter","final_iter"]].round(4), "Q1Q2_slope_by_arm", caption="Q1+Q2 per-iteration OLS slope + Spearman rho per arm (climb rate). Thin arms dropped.")
PCA = pd.DataFrame([{"arm": a, "PC1_pct": round(100*stats.rubric_pca(S.SCORES[S.SCORES.arm==a])["explained_variance_ratio"][0], 1)}
                    for a in sorted(S.SCORES.arm.unique()) if stats.rubric_pca(S.SCORES[S.SCORES.arm==a])["explained_variance_ratio"]])
display(PCA)
exp3.save_table(PCA, "rubric_pca_pc1", caption="Variance explained by PC1 of the rubric scores per arm (dominant PC1 => rubrics ~ one latent factor).")